# Track 1 generated artifact retention
Compress genuine prediction bundles, final NPZ/GLB, previews and provenance on CPU. Inspect prediction schemas; exclude pretrained weights and credentials.

In [ ]:
import sys,subprocess
from pathlib import Path
script=Path('/kaggle/working/archive_native_outputs.py');script.write_text('"""Compress actual episode artifacts on Kaggle CPU for GitHub publication.\n\nOnly authored output directories are read. Pretrained checkpoints, runtime caches\nand credentials are excluded. Prediction PTH schemas are checked before archiving.\n"""\nimport argparse\nimport gc\nimport hashlib\nimport json\nfrom pathlib import Path\nimport tarfile\nimport tempfile\n\nimport torch\n\n\nPREDICTIONS = {\'coconet.pth\', \'shared_identity.pth\', \'refined.pth\', \'refinement_checkpoint.pth\'}\nSCHEMAS = {\'cari4d.mhr_wild_inference.v1\', \'cari4d.mhr_postopt_smplh_parity.v10\'}\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with path.open(\'rb\') as stream:\n        while chunk := stream.read(8 * 1024 * 1024):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef prediction_schema(path):\n    if path.name not in PREDICTIONS:\n        raise ValueError(\'Unexpected PTH file in generated artifacts: \' + path.name)\n    payload = torch.load(path, map_location=\'cpu\', weights_only=False)\n    if not isinstance(payload, dict) or payload.get(\'schema\') not in SCHEMAS:\n        raise ValueError(\'Expected actual prediction/pose-optimization output, not pretrained weights\')\n    if any(key in payload for key in (\'state_dict\', \'model_state_dict\', \'model\')):\n        raise ValueError(\'Unexpected model weights in generated artifacts\')\n    result = {\'schema\': payload[\'schema\'], \'root_keys\': sorted(payload)}\n    del payload\n    gc.collect()\n    return result\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--input\', type=Path, default=Path(\'/kaggle/input\'))\n    p.add_argument(\'--output\', type=Path, default=Path(\'/kaggle/working\'))\n    a = p.parse_args()\n    roster = {}\n    for report in a.input.rglob(\'batch_reconstruction_report.json\'):\n        for row in json.loads(report.read_text()):\n            episode = row[\'episode\']\n            if row[\'success\'] is not True or episode not in range(30) or episode in roster:\n                raise ValueError(\'Require 30 unique successful actual episode runs\')\n            roster[episode] = (report.parent, row)\n    if set(roster) != set(range(30)):\n        raise ValueError(\'Incomplete actual roster\')\n    a.output.mkdir(parents=True, exist_ok=True)\n    archives = []\n    for episode, (source, run) in sorted(roster.items()):\n        stem = f\'episode_{episode:06d}\'\n        files = []\n        native = source / \'native_outputs\' / stem\n        if not native.is_dir():\n            raise FileNotFoundError(native)\n        for path in native.rglob(\'*\'):\n            if not path.is_file():\n                continue\n            if path.is_symlink() or not (path.suffix in {\'.pth\', \'.pkl\', \'.json\', \'.glb\'} or path.name.endswith(\'.sam3d.h5\')):\n                raise ValueError(\'Unexpected generated-output file: \' + str(path))\n            files.append((path, \'native/\' + path.relative_to(native).as_posix()))\n        for suffix in (\'.npz\', \'_object.glb\', \'.report.json\'):\n            path = source / \'episodes\' / (stem + suffix)\n            if not path.is_file():\n                raise FileNotFoundError(path)\n            files.append((path, \'episodes/\' + path.name))\n        for path in (source / \'previews\' / stem).glob(\'*.mp4\'):\n            files.append((path, \'previews/\' + path.name))\n        log = source / (stem + \'.log\')\n        if log.is_file():\n            files.append((log, log.name))\n        identities = {}\n        for path, destination in files:\n            item = {\'bytes\': path.stat().st_size, \'sha256\': digest(path)}\n            if path.suffix == \'.pth\':\n                item[\'prediction_inspection\'] = prediction_schema(path)\n            identities[destination] = item\n        archive_path = a.output / (stem + \'_native_artifacts.tar.gz\')\n        if archive_path.exists():\n            raise FileExistsError(archive_path)\n        manifest = {\'episode\': episode, \'run\': run, \'source_output\': str(source.relative_to(a.input)),\n                    \'pretrained_weights_included\': False, \'files\': identities}\n        with tempfile.TemporaryDirectory() as temporary:\n            metadata = Path(temporary) / \'manifest.json\'\n            metadata.write_text(json.dumps(manifest, indent=2) + \'\\n\')\n            with tarfile.open(archive_path, \'w:gz\', compresslevel=1) as archive:\n                archive.add(metadata, arcname=\'manifest.json\')\n                for path, destination in files:\n                    archive.add(path, arcname=destination, recursive=False)\n        record = {\'episode\': episode, \'archive\': archive_path.name,\n                  \'bytes\': archive_path.stat().st_size, \'sha256\': digest(archive_path),\n                  \'pretrained_weights_included\': False}\n        archives.append(record)\n        (a.output / \'native_artifact_manifest.json\').write_text(json.dumps(archives, indent=2) + \'\\n\')\n        print(\'NATIVE_ARTIFACT_ARCHIVED\', json.dumps(record), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n')
subprocess.run([sys.executable,str(script)],check=True)
